### Fetch Data

Pull lesson pages straight from the course repository, pinned to commit `8c1834d` so everyone works with the same data.

In [60]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

---

### Question 1 : Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

In [61]:
from embedder import Embedder
embed = Embedder()
q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)
v[0]

np.float64(-0.02058203437252893)

**Answer: -0.02**

---

### Question 2 : Cosine similarity

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

In [62]:
doc = [d for d in documents if d['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]
doc_vector = embed.encode(doc['content'])

In [63]:
cosine_sim = doc_vector.dot(v)
print(cosine_sim)

0.36107027225589694


**Answer: 0.361**

---

### Question 3 : Chunking and search by hand

Which file does the highest-scoring chunk belong to (its `filename`)?

In [64]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [65]:
X = embed.encode_batch([chunk['content'] for chunk in chunks])

In [66]:
scores = X.dot(v)
best_idx = scores.argmax()
print(chunks[best_idx]['filename'])

02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer: 02-vector-search/lessons/07-sqlitesearch-vector.md**

---

### Question 4 : Vector search with minsearch

Using `VectorSearch` from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

In [67]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [68]:
q4 = "What metric do we use to evaluate a search engine?"
v4 = embed.encode(q4)

results = vindex.search(v4, num_results=5)
print(results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


**Answer: 04-evaluation/lessons/05-search-metrics.md**

---

### Question 5 : Text search vs vector search

Vector search matches by meaning, keyword search by exact words. Compare both of this approach using the same chunks with `Index` from minsearch. Use `content` as a text field. Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

In [69]:
q5 = "How do I store vectors in PostgreSQL?"
v5 = embed.encode(q5)

vector_results = vindex.search(v5, num_results=5)

In [70]:
from minsearch import Index

tindex = Index(text_fields=["content"], keyword_fields=['filename'])
tindex.fit(chunks)
text_results = tindex.search(q5, num_results=5)

In [71]:
vector_files = set(r['filename'] for r in vector_results)
text_files = set(r['filename'] for r in text_results)

In [72]:
vector_files

{'02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md'}

In [73]:
text_files

{'02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md'}

In [74]:
# files shows up in the vector results but not in the text results
print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


**Answer: 02-vector-search/lessons/08-pgvector.md**

---

### Question 6 : Hybrid search

Run the query:

>"How do I give the model access to tools?" 

By using vector and text search and fuse the results with RRF, which file is ranked first?

In [75]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [78]:
q6 = "How do I give the model access to tools?"
v6 = embed.encode(q6)

vector_results = vindex.search(v6, num_results=5)
text_results = tindex.search(q6, num_results=5)

In [79]:
results = rrf([vector_results, text_results])
print(results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md


**Answer: 01-agentic-rag/lessons/13-function-calling.md**